# Cinder — Story-Iter Backend on Colab Pro

**Runtime:** GPU → A100 (40GB)  
**Estimated setup time:** ~10-15 min (model downloads)  
**Per-generation time:** ~10 min for 15 frames

Run cells **in order**. After Cell 4, copy the ngrok URL and paste it into `src/api/storyService.ts` as the `API_URL`.

In [ ]:
# ============================================================
# Cell 1: Verify GPU + Clone repo
# ============================================================
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

import os

# Clone the Cinder repo (replace with your GitHub repo URL)
REPO_URL = 'https://github.com/daniel-sit-cs/Cinder.git'  # <-- UPDATE THIS

if not os.path.exists('/content/Cinder'):
    !git clone {REPO_URL} /content/Cinder
else:
    print('Repo already cloned — pulling latest...')
    !cd /content/Cinder && git pull

# Add story-iter to Python path so colab_server.py can import StoryAdapterXL
import sys
sys.path.insert(0, '/content/Cinder/backend/story-iter')
print('✅ Repo ready.')

In [ ]:
# ============================================================
# Cell 2: Install dependencies
# ============================================================
# Install PyTorch with CUDA 12.1 first (Colab A100 needs cu121)
!pip install -q torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 \
    --index-url https://download.pytorch.org/whl/cu121

# Install remaining requirements
!pip install -q -r /content/Cinder/backend/requirements_colab.txt

print('✅ Dependencies installed.')

In [ ]:
# ============================================================
# Cell 3: Download model checkpoints from HuggingFace
# (~10 GB total — runs once, cache persists if using Drive)
# ============================================================
import os
from huggingface_hub import hf_hub_download, snapshot_download

os.chdir('/content/Cinder/backend')

os.makedirs('ckpt/story-ad', exist_ok=True)
os.makedirs('ckpt/ipa/sdxl_models', exist_ok=True)
os.makedirs('generated_videos', exist_ok=True)

# ---- RealVisXL_V4.0 base model (~7 GB) ----
print('Downloading RealVisXL_V4.0 (7 GB)...')
snapshot_download(
    repo_id='SG161222/RealVisXL_V4.0',
    local_dir='ckpt/story-ad/RealVisXL_V4',
    ignore_patterns=['*.ckpt', '*.safetensors.index.json'],
)
print('✅ RealVisXL downloaded.')

# ---- IP-Adapter SDXL weights (~500 MB) ----
print('Downloading IP-Adapter SDXL weights...')
hf_hub_download(
    repo_id='h94/IP-Adapter',
    filename='sdxl_models/ip-adapter_sdxl.bin',
    local_dir='ckpt/ipa',
)
print('✅ IP-Adapter weights downloaded.')

# ---- CLIP image encoder for IP-Adapter (~1.7 GB) ----
print('Downloading CLIP image encoder...')
snapshot_download(
    repo_id='h94/IP-Adapter',
    allow_patterns='sdxl_models/image_encoder/**',
    local_dir='ckpt/ipa',
)
print('✅ CLIP encoder downloaded.')
print('\n✅ All model checkpoints ready.')

In [ ]:
# ============================================================
# Cell 4: Upload secrets from Google Drive
#
# Before running this cell, upload these files to your Google Drive:
#   MyDrive/cinder-secrets/firebase-key.json
#   MyDrive/cinder-secrets/.env
#
# The .env file should contain:
#   REPLICATE_API_TOKEN=r8_...  (can be left blank for Story-Iter)
# ============================================================
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

secrets_src = '/content/drive/MyDrive/cinder-secrets'
backend_dir = '/content/Cinder/backend'

shutil.copy(f'{secrets_src}/firebase-key.json', f'{backend_dir}/firebase-key.json')
shutil.copy(f'{secrets_src}/.env',              f'{backend_dir}/.env')

print('✅ Secrets copied from Drive.')

In [ ]:
# ============================================================
# Cell 5: Start server + ngrok tunnel
#
# 1. Get a free ngrok auth token at https://dashboard.ngrok.com/
# 2. Paste it below
# 3. Run this cell — copy the printed ngrok URL
# 4. Update API_URL in src/api/storyService.ts with that URL
# ============================================================
import os, subprocess, time, sys
from pyngrok import ngrok, conf

NGROK_AUTH_TOKEN = '3ADGuQsDRp0Togx6QuG1oZ6o3Mr_4XVFA18BAXu4Q7iZk6EP1'  # <-- PASTE YOUR TOKEN

# Add story-iter to path so the server can import StoryAdapterXL
sys.path.insert(0, '/content/Cinder/NAVIS-main')
sys.path.insert(0, '/content/Cinder/NAVIS-main/ip_adapter')

os.chdir('/content/Cinder/backend')

conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Start uvicorn in background (model loading takes 2-3 min)
print('⏳ Starting Cinder Story-Iter server...')
# Pass PYTHONPATH so the subprocess can find ip_adapter
env = os.environ.copy()
env['PYTHONPATH'] = '/content/Cinder/NAVIS-main:/content/Cinder/NAVIS-main/ip_adapter:' + env.get('PYTHONPATH', '')
server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'colab_server:app',
     '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=env,
)

# Wait for server to load models
print('Waiting for models to load (~2-3 min)...')
import requests as req
for attempt in range(40):
    time.sleep(10)
    try:
        r = req.get('http://localhost:8000/health', timeout=5)
        if r.status_code == 200:
            print(f'✅ Server live after {(attempt+1)*10}s')
            break
    except Exception:
        print(f'   Still loading... ({(attempt+1)*10}s)')
else:
    print('❌ Server did not start in time. Check logs below.')
    out, _ = server_process.communicate(timeout=5)
    print(out.decode())

# Open ngrok tunnel
tunnel = ngrok.connect(8000)
NGROK_URL = tunnel.public_url

print(f'\n{'='*60}')
print(f'✅  NGROK URL: {NGROK_URL}')
print(f'{'='*60}')
print(f'\n👉 Update storyService.ts:')
print(f"   const API_URL = '{NGROK_URL}';")
print(f'\nAPI docs: {NGROK_URL}/docs')

In [ ]:
# ============================================================
# Cell 6 (optional): Test the endpoint directly from Colab
# ============================================================
import requests, json

test_payload = {
    'userId': 'colab-test',
    'prompt': 'a brave astronaut exploring an alien jungle',
    'style': 'comic',
    'frameCount': 3,         # Use 3 frames for a quick test
    'referenceImage': None,
}

print(f'Sending test request to {NGROK_URL}/generate-story...')
print('(This will take ~3-4 min for 3 frames)')

resp = requests.post(
    f'{NGROK_URL}/generate-story',
    json=test_payload,
    timeout=600,
)

if resp.status_code == 200:
    data = resp.json()
    print(f'\n✅ SUCCESS!')
    print(f'  Story ID : {data["storyId"]}')
    print(f'  Frames   : {len(data["frames"])}')
    print(f'  Video URL: {data["videoUrl"][:80]}...')
else:
    print(f'\n❌ Error {resp.status_code}: {resp.text[:500]}')